# B04. Reading the tree

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/b04-reading-the-tree/b04.ipynb)

The build you made in B01 came from about twelve hundred C files. You will open maybe thirty of them, ever.

So the useful skill is not knowing the tree, it is knowing three things about any file you land in: whether a person wrote it, whether you are allowed to change it, and why it says what it says. This lesson is those three, and it ends with you adding an instruction to CPython and watching the C for it appear.

![a table of the three kinds of file in the tree and what to do with each](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/three-kinds-of-file.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/opcode_ids.h:1-4@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Everything below was checked against the version this cell prints and against 3.14. Where the two disagree, the lesson says so.

In [ ]:
import pyxray

pyxray.show()

## Part of the tree is already on your machine

Before any of the C, the good news. Roughly half of CPython is written in Python, and the interpreter you are reading this on has all of it, on disk, in files you can open.

Better than that, it will tell you where. the running interpreter can hand you the file and the line range for anything in the standard library written in Python, in the same shape as the source references in this book, using two functions in `inspect`: [Lib/inspect.py:886-897@v3.15.0rc1#getsourcefile](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/inspect.py#L886-L897) gives you the file, and [Lib/inspect.py:1162-1181@v3.15.0rc1#getsourcelines](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/inspect.py#L1162-L1181) gives you the lines and the number of the first one.

![the two parts of a source reference, a file and a line range](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/a-reference-is-a-path.svg)

Put them together and you have made a reference of your own.

In [ ]:
import functools
import inspect
import json
import os
import textwrap
from pathlib import Path

# Where your standard library lives, asked of a module rather than of sysconfig. Both know,
# but a browser build serves the library out of a zip and sysconfig still reports the plain
# directory it would have been, while a module's own file is always the real answer.
LIB = Path(inspect.getsourcefile(textwrap)).parent


def reference(thing):
    """A file and a line range, in the same shape as the references in this lesson."""
    where = os.path.relpath(inspect.getsourcefile(thing), LIB)
    lines, first = inspect.getsourcelines(thing)
    return f"Lib/{where}:{first}-{first + len(lines) - 1}"


for thing in (functools.cache, json.JSONDecoder, textwrap.shorten, inspect.getsourcelines):
    print(f"{thing.__qualname__:18} {reference(thing)}")

> **Version note.** The line numbers are different on 3.14, because the files have been edited since. The point is that your own interpreter knows them, whatever they are.

Those are real line numbers in real files, and you can go one step further and read the thing itself. `functools.cache` is a decorator a lot of people use every week without ever having seen it, and it is three lines long.

In [ ]:
print(inspect.getsource(functools.cache))

That is the entire implementation. `cache` is `lru_cache` with the size limit taken off.

This works for anything written in Python and nothing written in C, and the reason is worth knowing rather than guessing: `getsourcefile` looks at the file the module was loaded from, and for a C module that is a `.so` with no source in it, so it returns `None` rather than pretending.

## An index of a file, in nine lines

The second move is turning a file into a list of what is in it. Editors do this and so does `grep`, but doing it yourself once is what makes the source stop feeling like a wall of text.

the ast module can list every function and class in a file with the exact line range of each, which is enough to build an index of any Python file in the standard library, and the whole of it is `ast.parse` followed by a loop over the top level.

In [ ]:
import ast

# inspect.getsource rather than reading the file, because inspect asks whoever imported the
# module for the text and that works even when the library is packed into a zip.
name = Path(inspect.getsourcefile(textwrap)).name
tree = ast.parse(inspect.getsource(textwrap))

print(f"Lib/{name}")
print()
for node in tree.body:
    if isinstance(node, ast.FunctionDef | ast.ClassDef):
        kind = "class" if isinstance(node, ast.ClassDef) else "def"
        print(f"  {kind:6} {node.name:14} Lib/{name}:{node.lineno}-{node.end_lineno}")

Six names, one file, and every one of them with a line range you could paste into a bug report.

Point that loop at a directory instead of a file and you have the index behind every jump to definition you have ever used. There is no more to it than this.

## The files nobody typed

Now the part that costs people afternoons.

Some files in the tree were not written by anyone. A script produced them while CPython was being built, and if you open one looking for an explanation you will not find one, because you are reading the output of a program rather than the program.

CPython is good about this. a generated file says so in its own first few lines, and usually names the script that wrote it, so you can find every one of them in your own standard library with a search for a handful of phrases. Here is [Include/opcode_ids.h:1-4@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Include/opcode_ids.h#L1-L4), which is four lines of banner before any code:

```c
// This file is generated by Tools/cases_generator/opcode_id_generator.py
// from:
//   Python/bytecodes.c
// Do not edit!
```

Your own standard library has a few of these. The next cell finds them.

In [ ]:
import re
import zipfile

SKIP = ("test/", "site-packages/", "idlelib/")
SAYS = re.compile(r"auto[- ]?generated|generated by|do not edit|don't edit", re.IGNORECASE)
NAMES_A_SCRIPT = re.compile(r"[\w/]+\.py")


def wanted(name):
    return name.endswith(".py") and not name.startswith(SKIP)


def everything():
    """Every Python file in your standard library, paired with the text of it.

    A browser build ships the whole library as a single zip rather than as a directory, so
    pathlib cannot walk it there. Two lines of zipfile covers both cases.
    """
    if LIB.suffix == ".zip":
        with zipfile.ZipFile(LIB) as bundle:
            names = [one for one in bundle.namelist() if wanted(one)]
            return [(one, bundle.read(one).decode("utf-8", "replace")) for one in names]
    on_disk = [one for one in LIB.rglob("*.py") if wanted(one.relative_to(LIB).as_posix())]
    return [
        (one.relative_to(LIB).as_posix(), one.read_text(encoding="utf-8", errors="replace"))
        for one in on_disk
    ]


def banner(text):
    """The first six lines, which is where a generated file owns up to being one."""
    return "\n".join(text.splitlines()[:6])


yours = everything()
generated = sorted((name, text) for name, text in yours if SAYS.search(banner(text)))

print(f"Python files in your standard library: {len(yours)}")
print(f"files that say a script wrote them:    {len(generated)}")
print()
for name, text in generated:
    #: Skip past the file's own name so a module called foo.py does not claim to have
    #: written itself.
    said = banner(text)[len(name.rpartition("/")[2]) :]
    wrote_it = NAMES_A_SCRIPT.search(said)
    print(f"  Lib/{name:26} {wrote_it.group(0) if wrote_it else 'does not say which script'}")

> **Version note.** How many files you have depends on the install rather than the version. A source build, a framework install and a Colab image all count differently, and some ship extra files of their own. The named ones in the middle are on all of them.

`Lib/token.py` is the one to look at. Every name in it, `NAME`, `NUMBER`, `INDENT`, the whole list T02 spent a lesson on, comes out of `Grammar/Tokens` through a script. Nobody has typed that file since 2018.

`Lib/_opcode_metadata.py` is the other interesting one, and it is the thread this lesson pulls on for the rest of the way. It is generated from `Python/bytecodes.c`, which is also where the C for the eval loop comes from.

That is your standard library, which is the small half. On the C side it is not a handful of files.

How much of the C in CPython was written by a script rather than by a person?

```python
"""Count how much of the C in a real CPython checkout was written by a script.

Z02 makes this claim and cannot check it, because checking it needs the whole source tree and
that lesson deliberately does not download one. The build image has the tree the interpreter
was compiled from sitting at /usr/src/cpython, so here the claim is a measurement.

The rule for spotting a generated file is the one from Z02 and nothing cleverer: the file says
so in its own first three lines. The part worth reading is the bottom block, where every one of
those files is asked which script wrote it, and answers.
"""

import re
import time
from collections import Counter
from pathlib import Path

TREE = Path("/usr/src/cpython")
MARKERS = ("generated", "do not edit", "autogenerated")

#: The banners are not one format. pegen writes `@generated by pegen from python.gram`, the
#: cases generator writes a path on a line of its own, asdl_c writes a sentence, and Argument
#: Clinic writes a marker with no script name in it at all. Three patterns cover every file.
WROTE_IT = re.compile(r"[A-Za-z_][\w/]*\.py|\bpegen\b|\[clinic input\]")
NAMES = {"pegen": "Parser/pegen, the parser generator", "[clinic input]": "Argument Clinic"}


def head(path):
    return path.read_text(encoding="utf-8", errors="replace").split("\n")[:3]


def looks_generated(lines):
    return any(marker in " ".join(lines).lower() for marker in MARKERS)


def who_wrote(lines):
    found = WROTE_IT.search(" ".join(lines))
    if found is None:
        return "it does not say"
    return NAMES.get(found.group(0), found.group(0))


started = time.monotonic()
files = sorted(p for p in TREE.rglob("*") if p.suffix in {".c", ".h"} and p.is_file())

lines_in_all = 0
generated = []
for path in files:
    top = head(path)
    lines = len(path.read_text(encoding="utf-8", errors="replace").splitlines())
    lines_in_all += lines
    if looks_generated(top):
        generated.append((lines, str(path.relative_to(TREE)), who_wrote(top)))
took = time.monotonic() - started

lines_generated = sum(count for count, _, _ in generated)

print(f"C and header files in the tree: {len(files):>10,}")
print(f"lines in them:                  {lines_in_all:>10,}")
print()
print(f"files a script wrote:           {len(generated):>10,}")
print(f"lines a script wrote:           {lines_generated:>10,}")
print(f"share of the C nobody typed:    {lines_generated / lines_in_all:>10.1%}")
print()
print("the eight biggest, and the script each one names in its own first three lines")
print()
for lines, where, script in sorted(generated, reverse=True)[:8]:
    print(f"  {lines:>7,}  {where:<43} {script}")

print()
print("who wrote the most files")
print()
counted = Counter(script for _, _, script in generated)
for script, many in counted.most_common(6):
    print(f"  {many:>4} files   {script}")

print()
print(f"~ how long the scan took, in seconds: {took:.1f}")

assert len(files) > 1000, len(files)
assert 0.3 < lines_generated / lines_in_all < 0.45
```

```text
C and header files in the tree:      1,185
lines in them:                   1,084,549

files a script wrote:                  237
lines a script wrote:              405,893
share of the C nobody typed:         37.4%

the eight biggest, and the script each one names in its own first three lines

   39,486  Parser/parser.c                             Parser/pegen, the parser generator
   24,443  Python/executor_cases.c.h                   Tools/cases_generator/tier2_generator.py
   18,524  Python/Python-ast.c                         Parser/asdl_c.py
   18,502  Modules/unicodename_db.h                    Tools/unicode/makeunicodedata.py
   13,645  Modules/clinic/posixmodule.c.h              Argument Clinic
   13,295  Modules/_testinternalcapi/test_cases.c.h    Tools/cases_generator/tier1_generator.py
   13,292  Python/generated_cases.c.h                  Tools/cases_generator/tier1_generator.py
    9,457  Modules/_ssl_data_36.h                      Tools/ssl/make_ssl_data.py

who wrote the most files

   164 files   Argument Clinic
    25 files   Programs/_freeze_module.py
    12 files   it does not say
     5 files   Tools/ssl/make_ssl_data.py
     3 files   Parser/asdl_c.py
     3 files   Tools/build/generate_slots.py

~ how long the scan took, in seconds: 0.5
```

That ran on Python 3.15.0rc1 in the debug build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824 python3 -` takes the program on standard input.

![a bar chart of generated against hand written lines of C](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/typed-and-not-typed.svg)

Four hundred thousand lines out of a million, and Argument Clinic alone wrote 164 files.

Two things in that list are worth a second look. `Programs/_freeze_module.py` wrote 25 files that are not in a fresh checkout at all, because freezing the startup modules happens during the build. And the twelve that say `it does not say` are almost all the Unicode tables, which are old enough to predate the convention.

Z02 makes the same claim about a third of the C being generated and has to mark it as something you cannot check, because checking it needs the whole tree. That is what the recording above is for. Same rule, same phrases, run against the tree the debug image was built from.

## The file that four other files come from

[Generated files](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#generated-file) are only annoying while you do not know where they came from. Once you do, they are the opposite: one file to read instead of four.

![a tree showing one input file producing four generated files through four scripts](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/one-file-many-files.svg)

`Python/bytecodes.c` is the source of truth for what every [instruction](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#instruction) does. Nothing else in CPython describes an instruction. The scripts in `Tools/cases_generator/` read it and write the eval loop, the opcode numbers, the jump table and the Python side of the `dis` module, and running those scripts against the unchanged input reproduces the committed files byte for byte, which is what makes generated a fact about a file rather than a comment in it.

The one that writes the opcode numbers is nine lines long. This is the whole of it, from [Tools/cases_generator/opcode_id_generator.py:24-41@v3.15.0rc1#generate_opcode_header](https://github.com/python/cpython/blob/v3.15.0rc1/Tools/cases_generator/opcode_id_generator.py#L24-L41):

```python
def generate_opcode_header(filenames, analysis, outfile):
    write_header(__file__, filenames, outfile)
    out = CWriter(outfile, 0, False)
    with out.header_guard("Py_OPCODE_IDS_H"):
        out.emit("/* Instruction opcodes for compiled code */\n")

        def write_define(name, op):
            out.emit(f"#define {name:<38} {op:>3}\n")

        for op, name in sorted([(op, name) for (name, op) in analysis.opmap.items()]):
            write_define(name, op)
```

A header guard, a comment, and a loop that prints one `#define` per instruction. That is where every opcode number in Python comes from.

So there is a thing you can do that sounds much harder than it is: add an instruction to `Python/bytecodes.c`, run the scripts, and read what they wrote. The next recording does exactly that, on the tree in the debug image, and nothing it does touches the tree itself.

What happens to the generated C when you add an instruction to Python/bytecodes.c?

```python
"""Run the generators behind a third of CPython's C, then change what they read.

Two halves. The first regenerates four files that are already in the tree and compares them
byte for byte with what is there, which is what turns the word `generated` from a comment into
a fact. The second adds three lines to a copy of Python/bytecodes.c and runs two of the same
generators again, so one new instruction becomes an opcode number and thirteen lines of C.

Nothing here touches the tree. Every output goes to /tmp and the build is left as it was.
"""

import subprocess
import sys
import time
from pathlib import Path

TREE = Path("/usr/src/cpython")
BYTECODES = TREE / "Python/bytecodes.c"
GENERATORS = TREE / "Tools/cases_generator"

#: Four of the twelve things `make regen-cases` runs, picked because between them they cover a
#: header of numbers, a jump table, a Python module and the body of the eval loop.
JOBS = [
    ("opcode_id_generator.py", "Include/opcode_ids.h"),
    ("target_generator.py", "Python/opcode_targets.h"),
    ("py_metadata_generator.py", "Lib/_opcode_metadata.py"),
    ("tier1_generator.py", "Python/generated_cases.c.h"),
]

#: The three lines being added, and the instruction they go after. NOP is the smallest thing in
#: the file, so copying its shape gives an instruction that takes nothing and returns nothing.
NOP = """        pure inst(NOP, (--)) {
        }
"""
ADDED = """        pure inst(SHOUT, (--)) {
            printf("this instruction was not here an hour ago\\n");
        }
"""


def run(generator, source, into):
    subprocess.run(
        [sys.executable, str(GENERATORS / generator), "-o", str(into), str(source)],
        check=True,
        capture_output=True,
    )
    return Path(into)


counted = len(BYTECODES.read_text().splitlines())
print(f"input:      Python/bytecodes.c, {counted:,} lines")
print("generators: Tools/cases_generator")
print()

started = time.monotonic()
for generator, output in JOBS:
    fresh = run(generator, BYTECODES, f"/tmp/{Path(output).name}")
    already = TREE / output
    same = fresh.read_bytes() == already.read_bytes()
    lines = len(already.read_text().splitlines())
    print(f"  {output:<32} {lines:>7,} lines   byte for byte identical: {same}")
    assert same, output
took = time.monotonic() - started

print()
print("Now the same generators, with three lines added to a copy of the input.")
print()
for line in ADDED.splitlines():
    print("   ", line.removeprefix("        "))

source = BYTECODES.read_text()
assert source.count(NOP) == 1
changed = Path("/tmp/bytecodes.c")
changed.write_text(source.replace(NOP, NOP + "\n" + ADDED))

ids = run("opcode_id_generator.py", changed, "/tmp/new_ids.h")
cases = run("tier1_generator.py", changed, "/tmp/new_cases.c.h")

print()
print("Include/opcode_ids.h comes back with a number for it:")
print()
for line in ids.read_text().splitlines():
    if "SHOUT" in line:
        print("   ", line.rstrip())

body = cases.read_text().splitlines()
at = body.index("        TARGET(SHOUT) {")
ends = body.index("        }", at)
print()
print("and Python/generated_cases.c.h comes back with the eval loop case for it:")
print()
for line in body[at : ends + 1]:
    print("   ", line.removeprefix("        "))

print()
print(f"~ how long the first four generators took, in seconds: {took:.1f}")

assert "#define SHOUT" in ids.read_text()
assert "this instruction was not here an hour ago" in cases.read_text()
```

```text
input:      Python/bytecodes.c, 6,725 lines
generators: Tools/cases_generator

  Include/opcode_ids.h                 266 lines   byte for byte identical: True
  Python/opcode_targets.h            1,295 lines   byte for byte identical: True
  Lib/_opcode_metadata.py              387 lines   byte for byte identical: True
  Python/generated_cases.c.h        13,292 lines   byte for byte identical: True

Now the same generators, with three lines added to a copy of the input.

    pure inst(SHOUT, (--)) {
        printf("this instruction was not here an hour ago\n");
    }

Include/opcode_ids.h comes back with a number for it:

    #define SHOUT                                   35

and Python/generated_cases.c.h comes back with the eval loop case for it:

    TARGET(SHOUT) {
        #if _Py_TAIL_CALL_INTERP
        int opcode = SHOUT;
        (void)(opcode);
        #endif
        frame->instr_ptr = next_instr;
        next_instr += 1;
        INSTRUCTION_STATS(SHOUT);
        _PyFrame_SetStackPointer(frame, stack_pointer);
        printf("this instruction was not here an hour ago\n");
        stack_pointer = _PyFrame_GetStackPointer(frame);
        DISPATCH();
    }

~ how long the first four generators took, in seconds: 3.8
```

That ran on Python 3.15.0rc1 in the debug build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:debug@sha256:de0d69b176872e99c63152b8ab4ebffac7bb131e5422c8eaebc7f455374ea824 python3 -` takes the program on standard input.

Read the last block twice. Three lines went into `bytecodes.c` and thirteen came out, and you wrote one of them.

`frame->instr_ptr = next_instr` and `next_instr += 1` are the interpreter moving along. `INSTRUCTION_STATS` is the counter T07 uses. The two `stack_pointer` lines are the eval loop putting the stack somewhere the rest of CPython can see it, because the C in the instruction body might call anything. `DISPATCH()` is the jump to the next instruction. Every instruction in CPython has that scaffolding around it and no instruction in `bytecodes.c` contains any of it.

That is the argument for the whole arrangement. The generated file is longer and duller than its input, which is exactly what you want from a machine.

If you were changing CPython for real, the command after the edit is `make regen-cases`, or `make regen-all` to rebuild every generated file at once. Forget it and the build quietly uses the old ones.

## Why is this line like this

The last skill is the one that stops you from breaking things.

Something in CPython will look wrong to you. A check that seems redundant, a branch that seems impossible, a comment that says "see the discussion" without saying where. Almost always somebody hit a real bug and this is the fix, and the way to find out is not to read harder.

![a table of four git commands and what each one answers about a line](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/four-ways-to-ask-why.svg)

The last two rows are the trick. every commit in CPython names an issue number in the first line of its message, so any line of code leads to a discussion: the format is `gh-NNNNNN: Summary of the changes made`, and the pull request template in the repository asks for it before anything else.

That number is the same number on the issue tracker, and the issue is where the argument happened. `git blame` gives you a commit, the commit gives you an issue, and the issue gives you three people disagreeing about the thing that is confusing you.

There is a second route to the same place that needs no git at all. Every user visible change ships a [blurb](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#blurb) file in `Misc/NEWS.d/`, and at release time they are collected into one file per version with the issue numbers kept. The next cell takes three real entries from [Misc/NEWS.d/3.15.0rc1.rst:11-33@v3.15.0rc1](https://github.com/python/cpython/blob/v3.15.0rc1/Misc/NEWS.d/3.15.0rc1.rst#L11-L33) and turns them into links.

In [ ]:
ENTRIES = """
.. date: 2026-08-02-11-47-15
.. gh-issue: 154902
.. nonce: DIi6sf
.. section: Core and Builtins

Fix a crash when ``__conditional_annotations__`` is rebound to a non-set
object.

..

.. date: 2026-07-28-10-00-00
.. gh-issue: 133931
.. nonce: fnQzxD
.. section: Core and Builtins

Fix data races when setting attributes of function objects on the
free threaded build.

..

.. date: 2026-07-27-16-29-10
.. gh-issue: 154775
.. nonce: _ISRIk
.. section: Core and Builtins

When matching a complex literal in case statements, an extraneous
``+`` sign (for example, ``1++1j`` or ``1-+1j``) is no longer allowed.
"""

TRACKER = "https://github.com/python/cpython/issues"

for block in ENTRIES.strip().split("\n..\n"):
    fields = dict(re.findall(r"^\.\. (\S+): (.+)$", block, re.MULTILINE))
    body = [line for line in block.splitlines() if line and not line.startswith("..")]
    print(f"{TRACKER}/{fields['gh-issue']}   {fields['section']}")
    print("   ", " ".join(body))
    print()

Three lines of parsing, and now every change in a release is a link to the argument behind it.

## Where the prose is

CPython has more written English in it than most people expect, and the trouble is that it is in five places written for five different readers.

![a table of five places CPython keeps prose and what each one is good for](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b04-reading-the-tree/diagrams/where-the-prose-is.svg)

`InternalDocs/` is the one worth knowing about, because it is the only one written for somebody in your position. It is short, it is honest about being incomplete, and it covers the parts of the interpreter that changed most recently, which are also the parts with the least written about them anywhere else.

The [devguide](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#devguide) is the other one. It is a separate repository, it is what a new contributor is pointed at, and it is where the build instructions, the test instructions and the etiquette live.

## Try it yourself

**One.** Point `reference` at something in a C module, like `math.sin` or `sys.getsizeof`, and read the error. Then work out from the source of `getsourcefile` why it happens.

**Two.** Change the `ast` cell to walk the whole tree with `ast.walk` rather than the top level, and print methods as well as functions. Then count how many of the names in `textwrap.py` start with an underscore.

**Three.** Run the generated file scan again with the `SKIP` set emptied out. Most of what appears is somebody else's package, which is a decent reminder that the convention is not CPython's alone.

**Four.** If you have a checkout, open `Python/bytecodes.c` and find `NOP` at line 148. It is two lines. Then find `TARGET(NOP)` in `Python/generated_cases.c.h` and count how much longer the generated version is.

**Five.** Pick any line in `Lib/textwrap.py` that looks odd to you and run `git blame` on it in a checkout. Follow the commit to its issue. This is the exercise that changes how you read the rest of these lessons.

## What just happened

Your own interpreter can hand you a file and a line range for anything written in Python, which is the same thing the references in this book are.

A file and `ast.parse` and nine lines gets you an index of that file, which is what a jump to definition is underneath.

A generated file says so in its first few lines and usually names the script. About a third of the C in CPython is that, and you saw the count made on a real tree rather than asserted.

`Python/bytecodes.c` is one file that four generated files come from, and you watched four lines added to it become an opcode number and twelve lines of eval loop.

Every commit names an issue, so any line of C leads to the discussion that put it there. That is the answer to nearly every why in this material.

## Where this goes next

That is the toolkit. A build, a debugger, a way to check nothing broke, and a way to read the tree and its history.

The lessons go back to the interpreter now, and they assume all four. When one of them says a number came from `Python/bytecodes.c`, you know what that file is and what happens to it. When one of them says a check was added for a reason, you know how to find the reason.